# db

> Fill in a module description here

**Torske Klubben** is a web application for managing a Norwegian-American social dining club based in Minnesota. The club holds 8 monthly luncheon meetings from October through May at the Edina Country Club (Social Hour at 11:00, Lunch at 12:00 Noon).

**Core features:**

- **Season schedule** — 8 event cards showing date, location, cost, reservation/cancellation deadlines, with the next upcoming event highlighted
- **Self-service reservations** — members can visit the site to reserve, edit, or decline for any event, and optionally add up to 2 named guests (for pre-made badges)
- **Managed invitations** — for less tech-savvy members, the admin emails invitations with encoded response links; members click to respond without needing a login
- **Payment tracking** — at-the-door payments via cash, check, or card, with cash-back and balance-due tracking per event
- **Admin dashboard** — member management, event CRUD, reservation oversight, guest tracking, payment recording, email composition (via Gmail API with markdown support), database backup/restore
- **Email system** — group-based email to club members and external contacts, with draft and send capability

**Tech stack:** FastHTML + MonsterUI + fastlite (SQLite) + Gmail API via solvemail

code we will want to clean:

```python
# === DATABASE SETUP ===
from fastlite import database
from datetime import datetime
import os, time

db_path = os.path.expanduser('~/repos/torskeklubben/data.db')
os.makedirs(os.path.dirname(db_path), exist_ok=True)
db = database(db_path)

# --- Core tables ---
members = db.t.members
events = db.t.events
reservations = db.t.reservations
invitations = db.t.invitations
guests = db.t.guests

if members not in db.t:
    members.create(id=int, name=str, email=str, pk='id')
    events.create(id=int, name=str, date=int, place=str, cost=int, description=str, map_link=str, deadline=int, pk='id')
    reservations.create(id=int, member_id=int, event_id=int, guests=int, paid=int, created_at=int, stripe_payment_intent=str, pk='id')
    invitations.create(id=int, event_id=int, member_id=int, status=str, pk='id')
    guests.create(id=int, member_id=int, event_id=int, guest_name=str, pk='id')

# --- Email group tables ---
if 'email_groups' not in db.t:
    db.t.email_groups.create(id=int, name=str, pk='id')
if 'email_group_members' not in db.t:
    db.t.email_group_members.create(id=int, group_id=int, member_id=int, pk='id')
email_groups = db.t.email_groups
email_group_members = db.t.email_group_members

# --- Contact tables ---
if 'contacts' not in db.t:
    db.t.contacts.create(id=int, first_name=str, last_name=str, email=str, pk='id')
if 'contact_groups' not in db.t:
    db.t.contact_groups.create(id=int, name=str, pk='id')
if 'contact_group_members' not in db.t:
    db.t.contact_group_members.create(id=int, group_id=int, contact_id=int, pk='id')
contacts = db.t.contacts
contact_groups = db.t.contact_groups
contact_group_members = db.t.contact_group_members

# --- Payments table ---
if 'payments' not in db.t:
    db.t.payments.create(
        id=int, event_id=int, member_id=int, total_lunches=int,
        total_owed=float, cash_amt=float, check_amt=float, card_amt=float,
        cash_back=float, balance_due=float, notes=str, date_paid=str, pk='id')
payments = db.t.payments

# --- Reservation changes table ---
if 'reservation_changes' not in db.t:
    db.t.reservation_changes.create(
        id=int, event_id=int, member_id=int,
        first_name=str, last_name=str,
        old_status=str, new_status=str,
        old_guests=int, new_guests=int,
        changed_by=str, changed_at=str, changed_at_unix=int,
        email_sent=int, pk='id')
reservation_changes = db.t.reservation_changes

# --- Table map (for generic editor/backup) ---
TABLE_MAP = {
    'members': members, 'events': events, 'reservations': reservations,
    'invitations': invitations, 'guests': guests, 'payments': payments,
    'reservation_changes': reservation_changes,
}
BACKUP_TABLES = list(TABLE_MAP.keys())

def get_columns(table_name):
    """Get column names for a table"""
    return [row[1] for row in db.execute(f"PRAGMA table_info({table_name})")]
```

```python
import mistletoe

def show_members():
    """List all members sorted by last name"""
    for m in sorted(members(), key=lambda m: (m['last_name'], m['first_name'])):
        suffix = f", {m['suffix']}" if m.get('suffix') else ""
        print(f"  [{m['id']:3d}] {m['last_name']}, {m['first_name']}{suffix}  ({m['email']})")

def show_member(id):
    """Show all fields for one member"""
    m = members[id]
    for k, v in m.items(): print(f"  {k}: {v}")

def edit_member(id, first_name=None, last_name=None, suffix=None, email=None):
    """Edit any fields, rebuild display name, save"""
    m = members[id]
    if first_name is not None: m['first_name'] = first_name
    if last_name is not None: m['last_name'] = last_name
    if suffix is not None: m['suffix'] = suffix
    if email is not None: m['email'] = email
    m['name'] = f"{m['first_name']} {m['last_name']}" + (f", {m['suffix']}" if m.get('suffix') else "")
    members.update(m)
    print(f"✓ Updated: {m['name']} ({m['email']})")

def get_recipients(group_id):
    """Get recipient list based on group_id selection."""
    if group_id == 'all':
        return [{'email': m['email'], 'name': m['name']} for m in members() if m.get('email')]
    elif group_id.startswith('tk_'):
        gid = int(group_id[3:])
        gm_ids = {m['member_id'] for m in email_group_members() if m['group_id'] == gid}
        return [{'email': m['email'], 'name': m['name']} for m in members() if m['id'] in gm_ids and m.get('email')]
    elif group_id.startswith('ct_'):
        gid = int(group_id[3:])
        ct_ids = {m['contact_id'] for m in contact_group_members() if m['group_id'] == gid}
        return [{'email': contacts[cid]['email'], 'name': f"{contacts[cid]['first_name']} {contacts[cid]['last_name']}"} for cid in ct_ids]
    return []

def log_reservation_change(event_id, member_id, old_status, new_status, old_guests, new_guests, changed_by='member'):
    """Log a reservation change and send acknowledgement email."""
    m = members[member_id]
    e = events[event_id]
    now = datetime.now()
    reservation_changes.insert(
        event_id=event_id, member_id=member_id,
        first_name=m.get('first_name', ''), last_name=m.get('last_name', ''),
        old_status=old_status or '', new_status=new_status,
        old_guests=old_guests or 0, new_guests=new_guests or 0,
        changed_by=changed_by,
        changed_at=now.isoformat(), changed_at_unix=int(now.timestamp()),
        email_sent=0)
    change_id = list(reservation_changes())[-1]['id']
    deadline = e.get('deadline_date', 'TBD')
    no_cancel = e.get('no_cancel_date', '')
    if new_status == 'going':
        action_text = "Your reservation has been confirmed"
        if old_status and old_status != 'going':
            action_text = "Your reservation has been updated to **confirmed**"
    elif new_status == 'declined':
        action_text = "Your reservation has been changed to **Declined**. We are sorry you are unable to attend"
    else:
        action_text = f"Your reservation status has been changed to **{new_status}**"
    guest_text = f"\n\nNumber of guests: **{new_guests}**" if new_guests else ""
    deadline_text = f"\n\nYou may change your reservation up until **{deadline}**." if deadline else ""
    if no_cancel:
        deadline_text += f" Please note that cancellations after **{no_cancel}** are not permitted."
    body_md = f"""Dear {m.get('first_name', m.get('name', 'Member'))},

Thank you for your response regarding **{e['name']}**.

{action_text}.{guest_text}{deadline_text}

Best regards,
Torske Klubben"""
    try:
        if m.get('email'):
            body_html = mistletoe.markdown(body_md)
            solvemail.send(to=m['email'], subj=f"Torske Klubben - Reservation Update: {e['name']}", body=body_md, html=body_html)
            reservation_changes.update({'id': change_id, 'email_sent': 1})
            print(f"✓ Change logged & email sent to {m.get('email')} for {e['name']}")
        else:
            print(f"⚠️ Change logged but no email for {m.get('name')} (no email address)")
    except Exception as ex:
        print(f"⚠️ Change logged but email failed for {m.get('name')}: {ex}")

def ensure_invitations(event_id):
    """Create pending invitation for every member who doesn't have one for this event."""
    existing = {i['member_id'] for i in invitations(where=f"event_id={event_id}")}
    for m in members():
        if m['id'] not in existing:
            invitations.insert(event_id=event_id, member_id=m['id'], status='pending')
```

In [ ]:
#| default_exp db

In [ ]:
#| export
from fastcore.utils import *
from fastlite import *
from pathlib import Path
from fastmigrate import create_db, run_migrations, setup_logging
from tempfile import NamedTemporaryFile

In [ ]:
#| export
testing = False
root_dir = Path().home() / 'torske_site'

In [ ]:
root_dir

Path('/app/data/torske_site')

In [ ]:
testing = True

In [ ]:
#| export
def get_db():
    "Create a new or read in existing database tables for solveit app"
    if testing:
        db_path = NamedTemporaryFile(suffix=".db").name  # Can't use :memory: b/c fastmigrate does not support it (yet)
    else:
        db_path = '/app/data/torskeklubben/data.db'
    
    create_db(db_path)
    run_migrations(db_path, root_dir/'migrations/')
    
    db = database(db_path)
    return db
     

In [ ]:
#| export
db = get_db()

In [ ]:
create_mod(db,"../torske_site/db_dc.py")

In [ ]:
#| export
from torske_site.db_dc import *
from torske_site import db_dc
db.link_dcs(db_dc)

phew okay that was nice. what's next

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice progress! You've got the database setup with fastmigrate and dataclasses wired up. 

Looking at the original code you want to clean, the remaining pieces are:

1. **Utility functions** — `show_members`, `show_member`, `edit_member`
2. **Business logic** — `get_recipients`, `log_reservation_change`, `ensure_invitations`
3. **Backup helpers** — `TABLE_MAP`, `BACKUP_TABLES`, `get_columns`

Which of these would you like to tackle next?

In [ ]:
for m in [
    Member(first_name="Ole",  last_name="Hansen",   suffix="",    email="ole@example.com"),
    Member(first_name="Lars", last_name="Olson",     suffix="Jr.", email="lars@example.com"),
    Member(first_name="Sven", last_name="Nordquist", suffix="",    email="sven@example.com"),
]: db.t.member.insert(m)

list(db.t.member())

[Member(id=1, first_name='Ole', last_name='Hansen', suffix='', email='ole@example.com'),
 Member(id=2, first_name='Lars', last_name='Olson', suffix='Jr.', email='lars@example.com'),
 Member(id=3, first_name='Sven', last_name='Nordquist', suffix='', email='sven@example.com')]

In [ ]:
from datetime import datetime

for e in [
    Event(name="October Luncheon",  date=int(datetime(2025,10,15,12).timestamp()), place="Edina Country Club", cost=35, description="Season opener", map_link="", deadline=int(datetime(2025,10,10).timestamp())),
    Event(name="November Luncheon", date=int(datetime(2025,11,19,12).timestamp()), place="Edina Country Club", cost=35, description="Lutefisk special",  map_link="", deadline=int(datetime(2025,11,14).timestamp())),
    Event(name="December Luncheon", date=int(datetime(2025,12,17,12).timestamp()), place="Edina Country Club", cost=40, description="Holiday celebration", map_link="", deadline=int(datetime(2025,12,12).timestamp())),
]: db.t.event.insert(e)

list(db.t.event())

[Event(id=1, name='October Luncheon', date=1760529600, place='Edina Country Club', cost=35, description='Season opener', map_link='', deadline=1760054400),
 Event(id=2, name='November Luncheon', date=1763553600, place='Edina Country Club', cost=35, description='Lutefisk special', map_link='', deadline=1763078400),
 Event(id=3, name='December Luncheon', date=1765972800, place='Edina Country Club', cost=40, description='Holiday celebration', map_link='', deadline=1765497600)]

In [ ]:
db.t.invitation.insert(Invitation(event_id=1, member_id=1, status='pending'))
list(db.t.invitation())

[Invitation(id=1, event_id=1, member_id=1, status='pending')]

In [ ]:
import time

db.t.reservation.insert(Reservation(invitation_id=1, paid=0, created_at=int(time.time()), stripe_payment_intent=''))
list(db.t.reservation())

[Reservation(id=1, invitation_id=1, paid=0, created_at=1773437285, stripe_payment_intent='')]

In [ ]:
for g in [
    Guest(reservation_id=1, guest_name="Ingrid Hansen"),
    Guest(reservation_id=1, guest_name="Erik Larsen"),
]: db.t.guest.insert(g)

list(db.t.guest())

[Guest(id=1, reservation_id=1, guest_name='Ingrid Hansen'),
 Guest(id=2, reservation_id=1, guest_name='Erik Larsen')]

In [ ]:
@patch(as_prop=True) # This is where the number of guests is calculated
def num_guests(self:Reservation): return len(db.t.guest(where=f"reservation_id={self.id}"))

In [ ]:
res = db.t.reservation()[0]
res

Reservation(id=1, invitation_id=1, paid=0, created_at=1773437285, stripe_payment_intent='')

In [ ]:
res.num_guests

2

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

Do I have the functionality built into this app to cut off reservations at a certain time?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You've got a `deadline` field on `Event` (as a unix timestamp), so the data is there. But I don't see any enforcement logic yet — nothing currently checks whether `time.time() > event.deadline` before allowing a reservation.

Would you like to add a helper that checks deadline eligibility, or would you prefer to enforce it at the route/UI level?

If someone reserves after the deadline cutoff, I would like to have them sent a message saying that invitations are closed, that his name will be placed on a waiting list if they like. I would need to have a table that holds these invitations in the case that someone cancels. If someone cancels, then the first person on the cancellation list should be notified and asked if he wishes to take the opening due to the cancellation. Do you have any suggestions?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great idea — this is a classic waitlist pattern. Here's how I'd approach it:

**New table: `waitlist`**
- `id`, `event_id`, `member_id`, `created_at` (unix timestamp for FIFO ordering), `status` (e.g. `waiting`, `offered`, `accepted`, `expired`)

**The flow:**
1. **Reserve after deadline** → instead of creating a reservation, insert into `waitlist` with `status='waiting'`, and show/send the "invitations are closed, you're on the waitlist" message
2. **Someone cancels** → find the first `waitlist` entry (by `created_at`) with `status='waiting'` for that event, flip it to `offered`, and send them a notification with a response link (similar to your existing invitation links)
3. **Waitlisted member responds** → if they accept, create the reservation and set `status='accepted'`; if they decline or don't respond within some window, mark `expired` and move to the next person on the list

A couple of things to consider:
- **Offer expiry** — do you want a time limit for the waitlisted person to respond before it rolls to the next person? (e.g. 24 hours)
- **Multiple cancellations** — if two people cancel, should you offer to the top two waitlisters simultaneously, or strictly one-at-a-time?

Want to start by adding the migration for the `waitlist` table?

I am trying to think of the best way to do this. How about send a message to all those on the waiting list and the one who answers first gets the place. Or do you think giving the first in line a short period to reply?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Both approaches have merit — it depends on what fits the club's culture:

**Notify all, first-come-first-served:**
- ✅ Fastest way to fill the spot — no waiting around for one person to check their email
- ✅ Simpler logic — no need for expiry timers or cascading offers
- ❌ Can feel unfair to someone who was first on the waitlist but was slow to check email
- ❌ You need to handle the race condition (two people click "accept" at nearly the same time)

**Sequential with a short window (e.g. 24 hours):**
- ✅ Feels fair — rewards those who signed up earliest
- ✅ No race condition — only one person has the offer at a time
- ❌ Could take days to fill if people are slow to respond, especially with older members who may not check email frequently
- ❌ More complex — needs a timer/cron job to expire and cascade

Given that your members skew older and cancellations are likely happening close to the event, I'd lean toward **notify all**. Speed matters more than queue fairness when you're trying to fill a seat days before a luncheon. The race condition is easy to handle — the first accepted response creates the reservation, and anyone who clicks after gets a "sorry, spot already filled" message.

What do you think?

Some of the members do not check email ofter so I think sending a message that clearly states that all on the waiting list have been notified and the first to respond can have the place. Let's go that way.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good choice. Let's start with the migration for the `waitlist` table. What number is your latest migration file?

I don't know how to work github and would really appreciate some intense tutoring in this regard. Please give me the exact steps to do what is right and let me know what I am doing. I essentially want to back up the working present version before changes are made that may break it. I should be able to restore it quickly and also have a permanent backup by way of our system backup that is outside of github, the solveit backup. So I don't know what version I am working with, it may not even have been versioned yet since this will be my first use of github.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's take this one step at a time. First, let's find out where things stand — run this to check if your project already has git set up:

```bash
cd ~/torske_site && git status
```

This will tell us one of two things:
- **"not a git repository"** — means we need to initialize one (most likely)
- A list of files — means someone already set it up

Run that and let me know what you see. Then I'll explain what's happening at each step as we go.

In [ ]:
cd ~/torske_site && git status

[Errno 2] No such file or directory: '/app/data/torske_site && git status'
/app/data


cd ~/torske_site && git status
Output:





[Errno 2] No such file or directory: '/app/data/torske_site && git status'
/app/data

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Ah, the `cd` command in solveit needs to be on its own line. Let's split it into two steps:

```python
%cd ~/torske_site
```

Then:

```python
!git status
```

Run the `%cd` first, then the `git status`.

In [ ]:
%cd ~/torske_site
git status

SyntaxError: invalid syntax (3617968710.py, line 2)

In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../migrations/0001-initialize.sql
	modified:   00_db.ipynb
	modified:   ../pyproject.toml
	modified:   ../torske_site/_modidx.py
	modified:   ../torske_site/db.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../torske_site/db_dc.py

no changes added to commit (use "git add" and/or "git commit -a")


!git status
Output:





On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../migrations/0001-initialize.sql
	modified:   00_db.ipynb
	modified:   ../pyproject.toml
	modified:   ../torske_site/_modidx.py
	modified:   ../torske_site/db.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../torske_site/db_dc.py

no changes added to commit (use "git add" and/or "git commit -a")

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Great news — you already have git set up and you're on the `main` branch, connected to a remote (`origin/main`). So the foundation is there.

Here's what git is telling us:

- **Modified files** — 5 files you've changed since the last commit (save point)
- **Untracked files** — 1 new file (`db_dc.py`) that git doesn't know about yet

What we want to do is create a **commit** — think of it as a snapshot/save point of your current working code that you can always get back to. Two steps:

1. **Stage** the files (tell git which files to include in the snapshot)
2. **Commit** them (actually save the snapshot with a description)

Let's start by staging everything:

```python
!git add -A
```

This tells git "include all changes and new files." Run that and we'll check what it looks like before committing.

In [ ]:
!git add -A